In [1]:
!pip install tensorly --quiet
!pip install timm --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 60.7 MB/s eta 0:00:0000:0100:01


In [ ]:
"""
================================================================================
NOTEBOOK 3 — FedProx  |  DenseNet-121  |  T4x2  (Updated)
================================================================================
FedProx (μ=0.01) — no compression, full model updates.
Backbone: DenseNet-121 (8M) via timm — same as NB9/NB10 for fair comparison.

CHANGES FROM ORIGINAL:
  • Model    : torchvision EfficientNet-B0 → timm DenseNet-121
  • BatchNorm: FREEZE_BN_IN_FL=True (same as NB9)
  • Batch    : 64 eval / 24 client (DenseNet heavier than B0)
  • evaluate(): DataParallel unwrap guard added (defensive)

OUTPUT: /kaggle/working/results/results_fedprox.pkl
================================================================================
"""

import os, random, json, pickle, time, gc
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from dataclasses import dataclass, field, asdict
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

try:
    from torch.amp import autocast, GradScaler
    _AMP_NEW = True
except ImportError:
    from torch.cuda.amp import autocast, GradScaler
    _AMP_NEW = False

import torchvision.transforms as transforms

try:
    import timm
    print(f"timm {timm.__version__} already installed")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "timm>=0.9.0"])
    import timm

from typing import List
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, roc_auc_score, f1_score,
                             precision_score, recall_score, confusion_matrix)
from tqdm.auto import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

print("=" * 70)
print("📓 NB3 — FedProx  |  DenseNet-121  |  T4x2")
print("=" * 70)
print(f"PyTorch: {torch.__version__} | timm: {timm.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")


# =============================================================================
# CONFIGURATION
# =============================================================================

@dataclass
class Config:
    UNIFIED_CSV: str = "/kaggle/input/datasets/ariroyjit/unified-dr-dataset/unified_dataset.csv"
    STATS_JSON:  str = "/kaggle/input/datasets/ariroyjit/unified-dr-dataset/dataset_stats.json"
    SAVE_DIR:    str = "/kaggle/working/results"

    TRAIN_RATIO: float = 0.70
    VAL_RATIO:   float = 0.15
    TEST_RATIO:  float = 0.15

    # ── Model ────────────────────────────────────────────────────────────────
    MODEL_NAME:  str   = "densenet121"
    TIMM_NAME:   str   = "densenet121"
    PRETRAINED:  bool  = True
    DROPOUT:     float = 0.4

    # ── Training ─────────────────────────────────────────────────────────────
    BATCH_SIZE:      int   = 64       # Eval/val/test batch size
    CLIENT_BATCH:    int   = 24       # DenseNet heavier than B0
    FL_ROUNDS:       int   = 25
    LOCAL_EPOCHS:    int   = 1

    BACKBONE_LR:     float = 3e-5
    HEAD_LR:         float = 3e-4
    WEIGHT_DECAY:    float = 1e-4
    GRAD_CLIP:       float = 1.0

    FL_FREEZE_ROUNDS: int  = 3
    FREEZE_BN_IN_FL:  bool = True

    POS_WEIGHT:      float = 2.0
    FEDPROX_MU:      float = 0.01

    SAVE_CHECKPOINTS: bool = True
    CHECKPOINT_EVERY: int  = 5

    NUM_CLIENTS:     int   = 5
    NON_IID_ALPHA:   float = 0.5

    SEEDS: List[int] = field(default_factory=lambda: [42, 123])

    DEVICE:      str  = "cuda" if torch.cuda.is_available() else "cpu"
    NUM_WORKERS: int  = 2
    PIN_MEMORY:  bool = True
    USE_AMP:     bool = True


config = Config()

# Auto-find CSV / stats
for _sp in [config.STATS_JSON,
            "/kaggle/input/unified-dr-dataset/dataset_stats.json",
            "/kaggle/input/datasets/ariroyjit/unified-dr-dataset/dataset_stats.json"]:
    if os.path.exists(_sp):
        with open(_sp) as f:
            _ds = json.load(f)
        config.STATS_JSON = _sp
        print(f"✅ Stats: {_sp}")
        break

for _cp in [config.UNIFIED_CSV,
            "/kaggle/input/unified-dr-dataset/unified_dataset.csv",
            "/kaggle/input/datasets/ariroyjit/unified-dr-dataset/unified_dataset.csv"]:
    if os.path.exists(_cp):
        config.UNIFIED_CSV = _cp
        print(f"✅ CSV: {_cp}")
        break

Path(config.SAVE_DIR).mkdir(parents=True, exist_ok=True)

print(f"\n  Model      : {config.MODEL_NAME}")
print(f"  Seeds      : {config.SEEDS}")
print(f"  FL Rounds  : {config.FL_ROUNDS}")
print(f"  FedProx μ  : {config.FEDPROX_MU}")
print(f"  Batch      : {config.BATCH_SIZE} eval / {config.CLIENT_BATCH} client")
print(f"  POS_WEIGHT : {config.POS_WEIGHT}")


# =============================================================================
# UTILITIES
# =============================================================================

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def worker_init_fn(worker_id):
    np.random.seed(torch.initial_seed() % 2**32 + worker_id)

def interpolate_history(history_list, eval_mask):
    result = list(history_list)
    eval_indices = [i for i in range(len(result)) if eval_mask[i]]
    if len(eval_indices) <= 1:
        return result
    for k in range(len(eval_indices) - 1):
        i_s, i_e = eval_indices[k], eval_indices[k+1]
        v_s, v_e, span = result[i_s], result[i_e], i_e - i_s
        for j in range(1, span):
            result[i_s + j] = v_s + (j / span) * (v_e - v_s)
    return result


# =============================================================================
# DATASET
# =============================================================================

def get_transforms(split='train'):
    try:
        with open(config.STATS_JSON) as f:
            st = json.load(f)
        mean, std = st['mean'], st['std']
    except Exception:
        mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

    norm = transforms.Normalize(mean, std)
    if split == 'train':
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
            transforms.ToTensor(), norm,
        ])
    return transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), norm])


class DRDataset(Dataset):
    def __init__(self, samples, transform=None):
        # Accepts either list of (path, label) tuples OR DataFrame
        if isinstance(samples, pd.DataFrame):
            self.samples = [(row['image_path'], int(row['binary_label']))
                            for _, row in samples.iterrows()]
        else:
            self.samples = samples
        self.transform = transform

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            img = Image.open(path).convert("RGB")
            if self.transform: img = self.transform(img)
            return img, torch.tensor(float(label), dtype=torch.float32)
        except Exception:
            return torch.zeros(3, 224, 224), torch.tensor(0.0)


def load_and_split_data(csv_path, seed=42):
    """
    FIXED — matches NB9 EXACTLY:
    • random_state=42 ALWAYS (not seed-dependent)
    • True 70/15/15 two-step stratified split
    • Path remapping for stale Kaggle dataset paths
    The seed parameter is accepted for API compatibility but IGNORED for splitting.
    """
    print(f"\n📂 Loading: {csv_path}")
    df = pd.read_csv(csv_path)

    # ── Path remapping (same logic as NB9) ───────────────────────────────────
    sample  = df['image_path'].iloc[:20].tolist()
    n_valid = sum(1 for p in sample if os.path.exists(p))
    if n_valid < 18:
        print(f"  ⚠️  Broken paths ({n_valid}/20) — scanning /kaggle/input/ ...")
        filename_to_real = {}
        if os.path.exists('/kaggle/input'):
            for slug in sorted(os.listdir('/kaggle/input')):
                for root, _, files in os.walk(f'/kaggle/input/{slug}'):
                    for fname in files:
                        if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                            if fname not in filename_to_real:
                                filename_to_real[fname] = os.path.join(root, fname)
        df = df.copy()
        df['image_path'] = df['image_path'].apply(
            lambda p: filename_to_real.get(os.path.basename(p), p)
                      if not os.path.exists(p) else p)
        n_fixed = sum(1 for p in df['image_path'].iloc[:20] if os.path.exists(p))
        print(f"  ✅ After remap: {n_fixed}/20 valid")

    # ── STEP 1: 70% train / 30% temp  — ALWAYS random_state=42 ─────────────
    # CRITICAL: must NOT use seed here — NB9 hardcodes 42 so every seed
    # trains/tests on exactly the same rows, making seed-to-seed variance
    # measure algorithm stochasticity rather than data sampling variation.
    train_df, temp_df = train_test_split(
        df,
        test_size=(1 - config.TRAIN_RATIO),       # 0.30
        stratify=df['binary_label'],
        random_state=42,                            # ← HARDCODED, not seed
    )

    # ── STEP 2: Split temp 50/50 → 15% val / 15% test ───────────────────────
    val_ratio_adj = config.VAL_RATIO / (config.VAL_RATIO + config.TEST_RATIO)  # 0.5
    val_df, test_df = train_test_split(
        temp_df,
        test_size=(1 - val_ratio_adj),             # 0.5
        stratify=temp_df['binary_label'],
        random_state=42,                            # ← HARDCODED, not seed
    )

    # Convert DataFrame rows to (path, label) tuples
    train_samples = [(row['image_path'], int(row['binary_label']))
                     for _, row in train_df.iterrows()]
    val_samples   = [(row['image_path'], int(row['binary_label']))
                     for _, row in val_df.iterrows()]
    test_samples  = [(row['image_path'], int(row['binary_label']))
                     for _, row in test_df.iterrows()]

    tr_pos = sum(s[1] for s in train_samples)
    print(f"  Train: {len(train_samples):,} (DR+: {tr_pos} = "
          f"{tr_pos/len(train_samples)*100:.1f}%)")
    print(f"  Val:   {len(val_samples):,}")
    print(f"  Test:  {len(test_samples):,}")

    return train_samples, val_samples, test_samples


def create_non_iid_clients(train_samples, num_clients=5, alpha=0.5, seed=42):
    """
    FIXED — matches NB9 EXACTLY:
    • Uses np.random.default_rng (NOT legacy np.random.seed)
    • Per-client val split is 10% (NB9) not 20% (old ablations)
    • Returns {cid: (train_ds, val_ds)} dict directly — same as NB9
    """
    rng = np.random.default_rng(seed)             # ← MATCHES NB9 (not legacy RNG)
    labels      = np.array([s[1] for s in train_samples])
    num_classes = len(np.unique(labels))
    client_indices = [[] for _ in range(num_clients)]

    for c in range(num_classes):
        class_idx = np.where(labels == c)[0]
        rng.shuffle(class_idx)                     # ← default_rng shuffle
        proportions = rng.dirichlet(alpha * np.ones(num_clients))
        proportions = (proportions * len(class_idx)).astype(int)
        proportions[-1] = len(class_idx) - proportions[:-1].sum()
        start = 0
        for cid, count in enumerate(proportions):
            client_indices[cid].extend(class_idx[start:start+count].tolist())
            start += count

    print(f"\n  Non-IID clients (α={alpha}):")
    client_datasets = {}
    for cid in range(num_clients):
        cs = [train_samples[i] for i in client_indices[cid]]
        cl = [s[1] for s in cs]
        pos = sum(cl)
        print(f"    C{cid}: {len(cs):,} | DR+: {pos} ({pos/max(len(cs),1)*100:.1f}%)")

        if len(cs) > 20:
            mc = min(sum(1 for l in cl if l == 0), sum(1 for l in cl if l == 1))
            if mc >= 2:
                c_tr, c_va = train_test_split(
                    cs, test_size=0.1,             # ← 10% like NB9 (was 20%)
                    stratify=[s[1] for s in cs],
                    random_state=seed + cid,
                )
            else:
                c_tr, c_va = train_test_split(
                    cs, test_size=0.1,
                    random_state=seed + cid,
                )
        else:
            c_tr, c_va = cs, cs[:0]

        client_datasets[cid] = (
            DRDataset(c_tr, get_transforms('train')),
            DRDataset(c_va, get_transforms('val')),
        )

    return client_datasets


class DRClassifier(nn.Module):
    def __init__(self, pretrained=True, dropout=0.4):
        super().__init__()
        self.backbone = timm.create_model(
            config.TIMM_NAME, pretrained=pretrained,
            num_classes=0, global_pool='avg',
        )
        # Detect feature_dim via dummy forward (reliable for all timm models)
        self.backbone.eval()
        with torch.no_grad():
            feature_dim = int(self.backbone(torch.zeros(1, 3, 224, 224)).shape[-1])
        print(f"  🔧 {config.MODEL_NAME}: feature_dim={feature_dim}")

        self.classifier = nn.Sequential(
            nn.Linear(feature_dim, 512), nn.ReLU(inplace=True),
            nn.Dropout(dropout), nn.Linear(512, 1),
        )
        n = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"  ✅ Trainable params: {n:,}")

    def forward(self, x):
        return self.classifier(self.backbone(x)).squeeze(-1)

    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = False

    def unfreeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = True

    def freeze_bn_for_fl(self):
        for m in self.modules():
            if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
                m.eval()
                for p in m.parameters(): p.requires_grad = False

    def get_param_groups(self, bb_lr, head_lr):
        return [{'params': self.backbone.parameters(), 'lr': bb_lr},
                {'params': self.classifier.parameters(), 'lr': head_lr}]


def create_model():
    m = DRClassifier(config.PRETRAINED, config.DROPOUT)
    return m

def create_client_model(device):
    return DRClassifier(pretrained=False, dropout=config.DROPOUT).to(device)


# =============================================================================
# EVALUATION  — always unwraps DataParallel (defensive)
# =============================================================================

def evaluate(model, loader, device, return_arrays=True):
    m = model.module if isinstance(model, nn.DataParallel) else model
    m.eval()
    yt, yp = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            probs = torch.sigmoid(m(x)).cpu().numpy()
            yp.extend(probs); yt.extend(y.numpy())
    yt = np.array(yt); yp = np.array(yp); yh = (yp > 0.5).astype(int)
    try: auc = roc_auc_score(yt, yp)
    except: auc = 0.5
    metrics = {'auc': auc,
               'f1': f1_score(yt, yh, zero_division=0),
               'precision': precision_score(yt, yh, zero_division=0),
               'recall': recall_score(yt, yh, zero_division=0),
               'accuracy': accuracy_score(yt, yh)}
    if return_arrays:
        metrics.update({'y_true': yt, 'y_prob': yp, 'y_pred': yh})
    return metrics


# =============================================================================
# LOCAL TRAINING (FedProx)
# =============================================================================

def local_train(client_model, train_loader, device, rnd, global_state, mu=0.01):
    init = {k: v.clone() for k, v in client_model.state_dict().items()}
    client_model.train()
    if config.FREEZE_BN_IN_FL: client_model.freeze_bn_for_fl()
    if rnd < config.FL_FREEZE_ROUNDS: client_model.freeze_backbone()
    else: client_model.unfreeze_backbone()

    gw = {k: v.clone().to(device) for k, v in global_state.items()
          if v.dtype in [torch.float32, torch.float16]}

    optimizer = optim.AdamW(
        client_model.get_param_groups(config.BACKBONE_LR, config.HEAD_LR),
        weight_decay=config.WEIGHT_DECAY)
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([config.POS_WEIGHT], device=device))
    if config.USE_AMP:
        scaler = GradScaler('cuda') if _AMP_NEW else GradScaler()

    for _ in range(config.LOCAL_EPOCHS):
        for x, y in train_loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            if config.USE_AMP:
                ctx = autocast('cuda') if _AMP_NEW else autocast()
                with ctx:
                    loss = criterion(client_model(x), y)
                    prox = sum(torch.norm(p - gw[n])**2
                               for n, p in client_model.named_parameters()
                               if n in gw and p.requires_grad)
                    loss = loss + (mu / 2.0) * prox
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(client_model.parameters(), config.GRAD_CLIP)
                scaler.step(optimizer); scaler.update()
            else:
                loss = criterion(client_model(x), y)
                prox = sum(torch.norm(p - gw[n])**2
                           for n, p in client_model.named_parameters()
                           if n in gw and p.requires_grad)
                (loss + (mu/2.0)*prox).backward()
                torch.nn.utils.clip_grad_norm_(client_model.parameters(), config.GRAD_CLIP)
                optimizer.step()

    final = client_model.state_dict()
    return {k: final[k] - init[k] for k in final}, len(train_loader.dataset)


def fedavg_aggregate(global_model, updates, device):
    state = global_model.state_dict()
    total = sum(u['n'] for u in updates)
    for key in state:
        if state[key].dtype in [torch.long, torch.int64]: continue
        ws = torch.zeros_like(state[key], dtype=torch.float32)
        for u in updates:
            ws += (u['n'] / total) * u['deltas'][key].float()
        state[key] = state[key] + ws.to(state[key].dtype)
    global_model.load_state_dict(state)


# =============================================================================
# FEDERATED TRAINING LOOP
# =============================================================================

def train_federated(client_datasets, val_loader, test_loader, seed=42):
    set_seed(seed)
    device = config.DEVICE
    t0     = time.time()

    # No DataParallel — freeze_bn_for_fl() causes CUBLAS crash with DP
    global_model = create_model().to(device)
    print(f"\n  🚀 FedProx [{config.MODEL_NAME}] seed={seed} | μ={config.FEDPROX_MU}")

    history    = {'val_auc': [], 'test_auc': [], 'comm_mb': [],
                  'comm_mb_cumulative': [], 'compression_ratio': []}
    eval_mask  = []
    total_comm = 0
    last_val   = 0.5; last_test = 0.5

    eval_rounds = {0, 5, 10, 15, 20, 24}
    print(f"  Eval rounds: {sorted(r+1 for r in eval_rounds)}")

    if config.SAVE_CHECKPOINTS:
        ckpt = Path(config.SAVE_DIR) / "checkpoints"
        ckpt.mkdir(parents=True, exist_ok=True)

    pbar = tqdm(range(config.FL_ROUNDS), desc=f"  FedProx seed={seed}")

    for rnd in pbar:
        updates    = []
        round_comm = 0
        global_state = global_model.state_dict()

        for cid, (train_ds, _) in client_datasets.items():
            dl = DataLoader(train_ds, batch_size=config.CLIENT_BATCH, shuffle=True,
                            num_workers=0,
                            generator=torch.Generator().manual_seed(seed+rnd+cid))
            cm = create_client_model(device)
            cm.load_state_dict(global_state)
            deltas, n = local_train(cm, dl, device, rnd, global_state, config.FEDPROX_MU)
            comm = sum(d.numel() * d.element_size() for d in deltas.values())
            round_comm += comm
            updates.append({'deltas': deltas, 'n': n})
            del cm; torch.cuda.empty_cache()

        fedavg_aggregate(global_model, updates, device)
        total_comm += round_comm
        del updates; torch.cuda.empty_cache()

        is_eval = (rnd in eval_rounds)
        if is_eval:
            val_m  = evaluate(global_model, val_loader,  device, return_arrays=False)
            test_m = evaluate(global_model, test_loader, device, return_arrays=False)
            last_val, last_test = val_m['auc'], test_m['auc']

        history['val_auc'].append(last_val)
        history['test_auc'].append(last_test)
        history['comm_mb'].append(round_comm / 1e6)
        history['comm_mb_cumulative'].append(total_comm / 1e6)
        history['compression_ratio'].append(1.0)
        eval_mask.append(is_eval)

        pbar.set_postfix({'val': f"{last_val:.4f}", 'test': f"{last_test:.4f}",
                          'MB': f"{total_comm/1e6:.0f}", '✓' if is_eval else '·': ''})

        if config.SAVE_CHECKPOINTS and (rnd + 1) % config.CHECKPOINT_EVERY == 0:
            torch.save({'round': rnd+1, 'model': global_model.state_dict(),
                        'history': history}, ckpt / f"fedprox_rnd{rnd+1}_s{seed}.pt")

    for key in ['val_auc', 'test_auc']:
        history[key] = interpolate_history(history[key], eval_mask)

    final   = evaluate(global_model, test_loader, device, return_arrays=True)
    elapsed = time.time() - t0
    print(f"\n  ✅ seed={seed} | AUC={final['auc']:.4f} | F1={final['f1']:.4f} "
          f"| Recall={final['recall']:.4f} | Time={elapsed/60:.1f}min")

    del global_model; torch.cuda.empty_cache(); gc.collect()
    return {'final': final, 'history': history, 'total_comm_mb': total_comm/1e6}


# =============================================================================
# PLOTS
# =============================================================================

def create_plots(results, save_dir):
    Path(save_dir).mkdir(parents=True, exist_ok=True)
    res  = results['FedProx']['all_results']
    r0   = res[0]; h = r0.get('history', {})
    mean_auc = results['FedProx']['mean_auc']
    std_auc  = results['FedProx']['std_auc']
    aucs     = [r['auc'] for r in res]

    fig = plt.figure(figsize=(20, 10))
    gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.38, wspace=0.32)

    # AUC bar
    ax = fig.add_subplot(gs[0, 0])
    ax.bar(['FedProx\nDenseNet-121'], [mean_auc], yerr=[std_auc], capsize=12,
           color='#2ECC71', edgecolor='black', linewidth=2, alpha=0.88)
    ax.set_ylabel('Test AUC', fontweight='bold')
    ax.set_title('(a) Test AUC', fontweight='bold')
    ax.set_ylim([max(0.75, mean_auc-0.08), min(1.0, mean_auc+0.06)])
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.text(0, mean_auc+std_auc+0.005, f'{mean_auc:.4f}±{std_auc:.4f}',
            ha='center', va='bottom', fontweight='bold', fontsize=11)

    # All metrics
    ax = fig.add_subplot(gs[0, 1])
    mnames  = ['auc', 'f1', 'precision', 'recall', 'accuracy']
    mlabels = ['AUC', 'F1', 'Prec', 'Recall', 'Acc']
    mmeans  = [np.mean([r[m] for r in res]) for m in mnames]
    mstds   = [np.std([r[m]  for r in res]) for m in mnames]
    bcols   = ['#2ECC71','#3498DB','#E67E22','#E74C3C','#9B59B6']
    bars = ax.bar(range(5), mmeans, yerr=mstds, capsize=6, color=bcols,
                  edgecolor='black', linewidth=1.5, alpha=0.85)
    ax.set_xticks(range(5)); ax.set_xticklabels(mlabels, fontsize=10)
    ax.set_ylim([0.55, 1.0]); ax.set_ylabel('Score', fontweight='bold')
    ax.set_title('(b) All Metrics', fontweight='bold')
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    for bar, val in zip(bars, mmeans):
        ax.text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.01,
                f'{val:.3f}', ha='center', fontsize=9, fontweight='bold')

    # Per-seed
    ax = fig.add_subplot(gs[0, 2])
    slabels = [str(s) for s in config.SEEDS[:len(aucs)]]
    ax.bar(slabels, aucs, color='#2ECC71', edgecolor='black', linewidth=1.5, alpha=0.85)
    ax.axhline(np.mean(aucs), color='red', linestyle='--', linewidth=2,
               label=f'Mean={np.mean(aucs):.4f}')
    ax.set_xlabel('Seed'); ax.set_ylabel('Test AUC')
    ax.set_title('(c) Per-Seed Reproducibility', fontweight='bold')
    ax.legend(fontsize=10); ax.grid(axis='y', alpha=0.3, linestyle='--')

    # Convergence
    ax = fig.add_subplot(gs[1, 0])
    if h.get('test_auc'):
        rnds = range(1, len(h['test_auc'])+1)
        ax.plot(rnds, h['test_auc'], 'g-o', lw=2, markersize=3, label='Test')
        ax.plot(rnds, h['val_auc'],  'r-s', lw=2, markersize=3, label='Val')
    ax.set_xlabel('Round'); ax.set_ylabel('AUC')
    ax.set_title('(d) Convergence (Seed 42)', fontweight='bold')
    ax.legend(fontsize=10); ax.grid(alpha=0.3, linestyle='--')

    # AUC vs Comm
    ax = fig.add_subplot(gs[1, 1])
    if h.get('comm_mb_cumulative') and h.get('test_auc'):
        ax.plot(h['comm_mb_cumulative'], h['test_auc'], 's-', lw=2,
                color='#2ECC71', markersize=3)
    ax.set_xlabel('Cumul. Comm (MB)'); ax.set_ylabel('Test AUC')
    ax.set_title('(e) AUC vs Communication', fontweight='bold')
    ax.grid(alpha=0.3, linestyle='--')

    # Confusion Matrix
    ax = fig.add_subplot(gs[1, 2])
    if 'y_true' in r0 and 'y_pred' in r0:
        cm = confusion_matrix(r0['y_true'], r0['y_pred'])
        sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', ax=ax,
                    xticklabels=['No DR', 'DR'], yticklabels=['No DR', 'DR'],
                    annot_kws={'fontsize': 13, 'fontweight': 'bold'})
        ax.set_ylabel('True'); ax.set_xlabel('Predicted')
    ax.set_title('(f) Confusion Matrix', fontweight='bold')

    plt.suptitle(f'FedProx (μ={config.FEDPROX_MU}) — DenseNet-121 — T4x2',
                 fontweight='bold', fontsize=14, y=1.01)
    for fmt in ['png', 'pdf']:
        plt.savefig(f"{save_dir}/nb3_fedprox_densenet121.{fmt}", dpi=180, bbox_inches='tight')
    print(f"📊 nb3_fedprox_densenet121.png")
    plt.close()


# =============================================================================
# MAIN
# =============================================================================

def main():
    print(f"\n{'='*70}")
    print("🚀 NB3 — FedProx | DenseNet-121")
    print(f"{'='*70}")

    exp_results = []
    total_t0    = time.time()

    for si, seed in enumerate(config.SEEDS):
        print(f"\n{'─'*50}")
        print(f"  Seed {seed}  ({si+1}/{len(config.SEEDS)})")
        print(f"{'─'*50}")

        train_s, val_s, test_s = load_and_split_data(config.UNIFIED_CSV, seed)

        val_loader  = DataLoader(DRDataset(val_s,  get_transforms('val')),
                                 batch_size=config.BATCH_SIZE, shuffle=False,
                                 num_workers=config.NUM_WORKERS, pin_memory=config.PIN_MEMORY)
        test_loader = DataLoader(DRDataset(test_s, get_transforms('val')),
                                 batch_size=config.BATCH_SIZE, shuffle=False,
                                 num_workers=config.NUM_WORKERS, pin_memory=config.PIN_MEMORY)

        # create_non_iid_clients returns {cid: (train_ds, val_ds)} directly
        # (fixed: uses default_rng + 10% client val split, matching NB9)
        client_ds = create_non_iid_clients(
            train_s, config.NUM_CLIENTS, config.NON_IID_ALPHA, seed)

        res = train_federated(client_ds, val_loader, test_loader, seed=seed)
        res['final']['history']       = res['history']
        res['final']['total_comm_mb'] = res['total_comm_mb']
        exp_results.append(res['final'])
        print(f"  📊 AUC:{res['final']['auc']:.4f} F1:{res['final']['f1']:.4f}")

    total_elapsed = time.time() - total_t0
    aucs  = [r['auc']       for r in exp_results]
    f1s   = [r['f1']        for r in exp_results]
    precs = [r['precision'] for r in exp_results]
    recs  = [r['recall']    for r in exp_results]

    results = {
        'FedProx': {
            'mean_auc':  np.mean(aucs), 'std_auc': np.std(aucs),
            'mean_comm': np.mean([r['total_comm_mb'] for r in exp_results]),
            'all_results': exp_results,
        }
    }

    print(f"\n{'='*70}")
    print("📊 FedProx | DenseNet-121 — RESULTS")
    print(f"{'='*70}")
    print(f"  AUC       : {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")
    print(f"  F1        : {np.mean(f1s):.4f}")
    print(f"  Precision : {np.mean(precs):.4f}")
    print(f"  Recall    : {np.mean(recs):.4f}")
    print(f"  Comm      : {results['FedProx']['mean_comm']:.1f} MB")
    print(f"  Time      : {total_elapsed/60:.1f} min")

    pkl_out = f"{config.SAVE_DIR}/results_fedprox.pkl"
    with open(pkl_out, 'wb') as f:
        pickle.dump(results, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"\n💾 Saved → {pkl_out}")

    create_plots(results, config.SAVE_DIR)
    print(f"\n✅ NB3 DONE — FedProx | DenseNet-121")
    print(f"   AUC = {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")


if __name__ == "__main__":
    main()

timm 1.0.24 already installed
📓 NB3 — FedProx  |  DenseNet-121  |  T4x2
PyTorch: 2.9.0+cu126 | timm: 1.0.24
CUDA: True
  GPU 0: Tesla T4
  GPU 1: Tesla T4
✅ Stats: /kaggle/input/datasets/ariroyjit/unified-dr-dataset/dataset_stats.json
✅ CSV: /kaggle/input/datasets/ariroyjit/unified-dr-dataset/unified_dataset.csv

  Model      : densenet121
  Seeds      : [42, 123]
  FL Rounds  : 25
  FedProx μ  : 0.01
  Batch      : 64 eval / 24 client
  POS_WEIGHT : 2.0

🚀 NB3 — FedProx | DenseNet-121

──────────────────────────────────────────────────
  Seed 42  (1/2)
──────────────────────────────────────────────────

📂 Loading: /kaggle/input/datasets/ariroyjit/unified-dr-dataset/unified_dataset.csv
  Train: 11,656 (DR+: 5895 = 50.6%)
  Val:   2,498
  Test:  2,498

  Non-IID clients (α=0.5):
    C0: 1,368 | DR+: 968 (70.8%)
    C1: 1,668 | DR+: 1525 (91.4%)
    C2: 4,323 | DR+: 1450 (33.5%)
    C3: 3,742 | DR+: 1884 (50.3%)
    C4: 555 | DR+: 68 (12.3%)


model.safetensors:   0%|          | 0.00/32.3M [00:00<?, ?B/s]

  🔧 densenet121: feature_dim=1024
  ✅ Trainable params: 7,479,169

  🚀 FedProx [densenet121] seed=42 | μ=0.01
  Eval rounds: [1, 6, 11, 16, 21, 25]


  FedProx seed=42:   0%|          | 0/25 [00:00<?, ?it/s]

  🔧 densenet121: feature_dim=1024
  ✅ Trainable params: 7,479,169
